# Atropos: Getting Started

This notebook demonstrates the basics of using Atropos to estimate ROI for LLM pruning and quantization.

In [ ]:
import pandas as pd

from atropos import DeploymentScenario, estimate_outcome
from atropos.presets import SCENARIOS, STRATEGIES

## Using Built-in Presets

In [ ]:
# View available scenarios and strategies
print("Scenarios:")
for name, scenario in SCENARIOS.items():
    print(f"  {name}: {scenario.parameters_b}B params")

print("\nStrategies:")
for name, strategy in STRATEGIES.items():
    mem_red = strategy.memory_reduction_fraction * 100
    print(f"  {name}: {mem_red:.0f}% memory reduction")

In [ ]:
# Analyze a preset scenario with a preset strategy
scenario = SCENARIOS['medium-coder']
strategy = STRATEGIES['structured_pruning']

outcome = estimate_outcome(scenario, strategy)

print(f"Scenario: {outcome.scenario_name}")
print(f"Strategy: {outcome.strategy_name}")
print(f"Annual Savings: ${outcome.annual_total_savings_usd:,.0f}")
print(f"Break-even: {outcome.break_even_years*12:.1f} months")

## Creating Custom Scenarios

In [ ]:
# Define a custom deployment scenario
my_scenario = DeploymentScenario(
    name="my-production-api",
    parameters_b=70.0,
    memory_gb=48.0,
    throughput_toks_per_sec=35.0,
    power_watts=600.0,
    requests_per_day=100000,
    tokens_per_request=1500,
    electricity_cost_per_kwh=0.12,
    annual_hardware_cost_usd=50000.0,
    one_time_project_cost_usd=45000.0,
)

print(f"Created: {my_scenario.name}")
daily = my_scenario.requests_per_day * my_scenario.tokens_per_request
print(f"Daily tokens: {daily:,}")

In [ ]:
# Evaluate different strategies
results = []
for s_name, s in STRATEGIES.items():
    outcome = estimate_outcome(my_scenario, s)
    mem_red = (1 - outcome.optimized_memory_gb / outcome.baseline_memory_gb) * 100
    results.append({
        'Strategy': s_name,
        'Memory Reduction %': mem_red,
        'Annual Savings $': outcome.annual_total_savings_usd,
    })

df = pd.DataFrame(results)
df